# Занятие 6. Методы квантовой механики

In [ ]:
# 📥 Установка пакетов и библиотек
! pip install numpy
! pip install matplotlib
! pip install plotly
! pip install nbformat>=4.2.0 
! pip install rdkit

In [ ]:
# 🛠️ Загрузка необходимых пакетов в среду выполнения
import os
import sys
import numpy as np
import math
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Draw

# show() открывает просмотрщик ORCA прямо под ячейкой
REPO = "/home/jupyter/project/computational-chemistry-course"
sys.path.append(os.path.join(REPO, "tools"))
from orca_tools import show

In [ ]:
# 🐋 Инициализация ORCA
ORCA_DIR = "/opt/orca"
ORCA = os.path.join(ORCA_DIR, "orca")  # ORCA стабильно работает только если задать абсолютный путь
print("ORCA найдена:", os.path.isfile(ORCA))

# Библиотеки и переменные для ORCA
os.environ["PATH"] = ORCA_DIR + ":/opt/openmpi/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = ORCA_DIR + ":" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OMPI_MCA_plm"] = "isolated"

# Рабочая папка для расчётов
os.chdir(os.path.join(REPO, "Занятия", "Занятие_6"))

# 💡 Глава 1. Свойства как производные энергии

## 1.1 Адиабатическая энергия ионизации

На предыдущем занятии (Глава 1.2) энергию ионизации оценивали двумя способами: по теореме Купманса ($I \approx -\varepsilon_{\text{HOMO}}$) и как разность полных энергий катиона и нейтральной молекулы для **одной и той же** геометрии. Обе оценки относятся к **вертикальной** энергии ионизации предполагая, что геометрия молекулы не успевает измениться (принцип Франка-Кондона), и катион образуется в геометрии нейтральной молекулы.

Но для катиона геометрия нейтральной молекулы не является минимумом ППЭ, и катион релаксирует. Разность энергий двух этих минимумов называется **адиабатической** энергией ионизации:

$$ I_v = E(\mathrm{M^+};\, R_\mathrm{M}) - E(\mathrm{M};\, R_\mathrm{M}) \qquad\qquad I_a = E(\mathrm{M^+};\, R_\mathrm{M^+}) - E(\mathrm{M};\, R_\mathrm{M}) $$

Всегда $I_v \ge I_a$. Их разность $\lambda^+ = I_v - I_a$ - **энергия реорганизации** катиона (она также, например, появляется в теории Маркуса при расчёте скорости переноса электрона). Порог ионизации в эксперименте соответствует переходу между нулевыми колебательными уровнями, поэтому для сравнения с экспериментом к адиабатической энергии добавляют поправку ZPE:

$$ I_0 = I_a + \mathrm{ZPE}(\mathrm{M^+}) - \mathrm{ZPE}(\mathrm{M}) $$

In [ ]:
# NH3: оптимизация геометрии и частоты нейтральной молекулы
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "nh3_neutral")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP Opt Freq
%pal nprocs 24 end

* xyz 0 1
N  -0.000000  -0.000000  -0.006798
H   0.943040  -0.000000  -0.378734
H  -0.471520   0.816710  -0.378734
H  -0.471520  -0.816710  -0.378734
*
"""
with open("nh3_neutral.inp", "w") as f:
    f.write(input_file)

!{ORCA} nh3_neutral.inp > nh3_neutral.out --oversubscribe

In [ ]:
# NH3+: одноточечный расчёт катиона в геометрии нейтральной молекулы
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "nh3_cation_sp")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP
%pal nprocs 24 end

* xyzfile 1 2 ../nh3_neutral/nh3_neutral.xyz
"""
with open("nh3_cation_sp.inp", "w") as f:
    f.write(input_file)

!{ORCA} nh3_cation_sp.inp > nh3_cation_sp.out --oversubscribe

In [ ]:
# NH3+: оптимизация геометрии катиона
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "nh3_cation")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP Opt Freq
%pal nprocs 24 end

* xyz 1 2
N   0.000002   0.000000  -0.285770
H   1.025680   0.000000  -0.288460
H  -0.512840   0.888269  -0.284385
H  -0.512841  -0.888269  -0.284385
*
"""
with open("nh3_cation.inp", "w") as f:
    f.write(input_file)

!{ORCA} nh3_cation.inp > nh3_cation.out --oversubscribe

In [ ]:
show("nh3_cation.out")

In [ ]:
hartree_2_ev = 27.2114

NH3             = {"E": None, "ZPE": None}    # nh3_neutral
NH3_cation_vert = {"E": None}                 # nh3_cation_vert
NH3_cation      = {"E": None, "ZPE": None}    # nh3_cation_opt

I_vert = (NH3_cation_vert["E"] - NH3["E"]) * hartree_2_ev
I_ad   = (NH3_cation["E"] - NH3["E"]) * hartree_2_ev
I_0    = I_ad + (NH3_cation["ZPE"] - NH3["ZPE"]) * hartree_2_ev
lam    = I_vert - I_ad

print(f"Вертикальная I_v          : {I_vert:6.2f} эВ")
print(f"Адиабатическая I_a        : {I_ad:6.2f} эВ")
print(f"Адиабатическая с ZPE, I_0 : {I_0:6.2f} эВ")
print(f"Энергия реорганизации λ⁺  : {lam:6.2f} эВ")

**Аммиак**. Экспериментальные данные:

| Величина | Эксперимент, эВ |
|---|---:|
| $I_0$ (адиабатическая) | 10.070 ± 0.020 |
| $I_v$ (вертикальная) | ≈ 10.8 |

https://webbook.nist.gov/cgi/cbook.cgi?ID=C7664417&Mask=20

## 1.2 Сродство к протону и газофазная основность

**Сродство к протону** (PA, proton affinity) - энтальпия, выделяющаяся при присоединении протона в газовой фазе, а **газофазная основность** (GB, gas-phase basicity) - соответствующая энергия Гиббса:

$$ \mathrm{B + H^+ \rightarrow BH^+} \qquad\qquad PA = -\Delta_r H^\circ(298) \qquad\qquad GB = -\Delta_r G^\circ(298) $$

Это самая простая кислотно-основная характеристика, поскольку в модели нет растворителя, нет противоиона, и расчёт сводится к вычислению энергий двух молекул (протонированной и депротонированной формы), потому что у протона нет электронов, колебаний и вращений, и его вклад известен точно (298.15 K, 1 атм):

$$ H^\circ(\mathrm{H^+}) = \tfrac{5}{2}RT = 6.2\ \text{кДж/моль} $$
$$ \qquad S^\circ(\mathrm{H^+}) = 108.9\ \text{Дж/(моль}\cdot\text{K)} \qquad \text{(по уравнению  Закура-Тетроде)}$$
$$ G^\circ(\mathrm{H^+}) = H^\circ - TS^\circ = -26.3\ \text{кДж/моль} $$

In [ ]:
amines = {
    # основания                 сопряжённые кислоты
    "NH3":     "N",             "NH4+":    "[NH4+]",
    "MeNH2":   "CN",            "MeNH3+":  "C[NH3+]",
    "Me2NH":   "CNC",           "Me2NH2+": "C[NH2+]C",
    "Me3N":    "CN(C)C",        "Me3NH+":  "C[NH+](C)C",
}

bases   = [k for k in amines if not k.endswith("+")]
cations = [k for k in amines if k.endswith("+")]
order   = bases + cations

mols = [Chem.MolFromSmiles(amines[k]) for k in order]
Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220, 180), legends=order)

In [ ]:
# Амины и их протонированные формы
# Оптимизация + частоты
main_path = os.path.join(REPO, "Занятия", "Занятие_6")

species = {
    # папка            заряд  геометрия
    "nh4_cation": (1, """\
N   -0.000001   0.000005  -0.000012
H   -0.668197  -0.257732  -0.743642
H    0.110872   1.026210   0.022532
H   -0.356605  -0.327695   0.911757
H    0.913932  -0.440788  -0.190636
"""),
    "menh2": (0, """\
C   -0.559316   0.072960  -0.026007
N    0.865217  -0.070308  -0.295531
H   -1.124455  -0.681362  -0.598954
H   -0.907278   1.059326  -0.376214
H   -0.869309  -0.024542   1.037767
H    1.197721  -0.979418   0.027912
H    1.397420   0.623344   0.231026
"""),
    "menh3_cation": (1, """\
C   -0.762800   0.081539  -0.013154
N    0.735722  -0.078615   0.012676
H   -1.175578  -0.616698  -0.752113
H   -0.998295   1.116386  -0.291779
H   -1.155019  -0.143956   0.986524
H    1.016219  -1.037233   0.270315
H    1.179343   0.556953   0.693320
H    1.160408   0.121627  -0.905793
"""),
    "me2nh": (0, """\
C    1.204667   0.158061   0.121502
N   -0.012250   0.227539  -0.662279
C   -1.215898  -0.068301   0.088401
H    1.366260  -0.813631   0.644925
H    1.196472   0.942442   0.898949
H    2.080521   0.344987  -0.520679
H    0.056984  -0.393386  -1.467091
H   -1.210438  -1.056078   0.606751
H   -2.094081  -0.041577  -0.576839
H   -1.372236   0.699945   0.866358
"""),
    "me2nh2_cation": (1, """\
C    1.190429   0.487812  -0.023819
N    0.114512  -0.557971  -0.106158
C   -1.281532  -0.044728   0.107879
H    2.165191   0.012336  -0.194352
H    0.999734   1.248783  -0.791630
H    1.159959   0.944038   0.974088
H    0.311026  -1.299354   0.581786
H    0.166203  -1.024742  -1.023204
H   -1.500913   0.706243  -0.662063
H   -1.338262   0.410670   1.105006
H   -1.986345  -0.883089   0.032464
"""),
    "me3n": (0, """\
C   -0.299748   1.331429   0.272151
N   -0.127668  -0.099573   0.410470
C    1.258568  -0.505828   0.311961
C   -0.993459  -0.852707  -0.472519
H   -0.008265   1.718906  -0.734364
H    0.310042   1.862747   1.021310
H   -1.353977   1.606422   0.440585
H    1.709972  -0.305922  -0.690160
H    1.353900  -1.586282   0.509262
H    1.865857   0.027386   1.061836
H   -0.774701  -0.688622  -1.555710
H   -0.893898  -1.932542  -0.273838
H   -2.046622  -0.575416  -0.300985
"""),
    "me3nh_cation": (1, """\
C    0.032569   1.414814   0.263332
N    0.023157   0.049609  -0.364312
C    1.219877  -0.763329   0.041442
C   -1.265000  -0.678780  -0.103871
H    0.966976   1.925597  -0.002604
H   -0.034825   1.298827   1.353138
H   -0.828433   1.985771  -0.107344
H    0.087301   0.188190  -1.381706
H    2.132752  -0.212207  -0.218598
H    1.196055  -1.725065  -0.487049
H    1.177896  -0.930055   1.125956
H   -1.363182  -0.839693   0.977872
H   -2.101644  -0.069839  -0.469891
H   -1.243502  -1.643838  -0.626364
"""),
}

for name, (charge, xyz) in species.items():
    work_path = os.path.join(main_path, name)
    os.makedirs(work_path, exist_ok=True)
    os.chdir(work_path)

    input_file = f"""\
! B3LYP def2-SVP Opt Freq
%pal nprocs 24 end

* xyz {charge} 1
{xyz}*
"""
    with open(f"{name}.inp", "w") as f:
        f.write(input_file)

    print("Расчёт:", name)
    !{ORCA} {name}.inp > {name}.out --oversubscribe

os.chdir(main_path)
# 2m 16s

In [ ]:
def thermo_from_out(path):
    """E(ele), H, G (хартри) из последнего блока THERMOCHEMISTRY файла .out"""
    labels = {
        "Electronic energy":       "E(ele)",
        "Total Enthalpy":          "H",
        "Final Gibbs free energy": "G",
    }
    values = {"E(ele)": None, "H": None, "G": None}
    finished = False
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            for prefix, key in labels.items():
                if line.startswith(prefix) and "..." in line:
                    values[key] = float(line.split("...")[1].split()[0])   # число перед "Eh"
            if "ORCA TERMINATED NORMALLY" in line:
                finished = True
    if not finished:
        print(f"{os.path.basename(path)} не завершён")
    return values

folders = {
    "NH3":     "nh3_neutral",
    "NH4+":    "nh4_cation",
    "MeNH2":   "menh2",
    "MeNH3+":  "menh3_cation",
    "Me2NH":   "me2nh",
    "Me2NH2+": "me2nh2_cation",
    "Me3N":    "me3n",
    "Me3NH+":  "me3nh_cation",
}

thermo = {}
print("thermo = {")
for label, folder in folders.items():
    out = os.path.join(main_path, folder, f"{folder}.out")
    if not os.path.isfile(out):
        print(f"    # {label}: нет файла {folder}/{folder}.out")
        continue
    thermo[label] = v = thermo_from_out(out)
    print(f'    "{label}":'.ljust(15) + f'{{"E(ele)": {v["E(ele)"]}, "H": {v["H"]}, "G": {v["G"]}}},   # {folder}')
print("}")

In [ ]:
hartree_2_kj = 2625.4996
T = 298.15
R = 8.314462e-3                        # кДж/(моль·K)
H_proton = 2.5 * R * T                 # 6.2 кДж/моль
S_proton = 0.10885                     # кДж/(моль·K)
G_proton = H_proton - T * S_proton     # -26.3 кДж/моль

thermo = {
    "NH3":     {"E(ele)": None, "H": None, "G": None},   # nh3_neutral
    "NH4+":    {"E(ele)": None, "H": None, "G": None},   # nh4_cation
    "MeNH2":   {"E(ele)": None, "H": None, "G": None},   # menh2
    "MeNH3+":  {"E(ele)": None, "H": None, "G": None},   # menh3_cation
    "Me2NH":   {"E(ele)": None, "H": None, "G": None},   # me2nh
    "Me2NH2+": {"E(ele)": None, "H": None, "G": None},   # me2nh2_cation
    "Me3N":    {"E(ele)": None, "H": None, "G": None},   # me3n
    "Me3NH+":  {"E(ele)": None, "H": None, "G": None},   # me3nh_cation
}

pairs = [("NH3", "NH4+"), ("MeNH2", "MeNH3+"), ("Me2NH", "Me2NH2+"), ("Me3N", "Me3NH+")]

# Hunter, Lias, J. Phys. Chem. Ref. Data 1998, 27, 413: (PA, GB), кДж/моль
exp = {"NH3": (853.6, 819.0), "MeNH2": (899.0, 864.5), "Me2NH": (929.5, 896.5), "Me3N": (948.9, 918.1)}

print(f"{'основание':<10}{'ΔE_el':>8}{'PA':>8}{'GB':>8}{'PA эксп':>9}{'GB эксп':>9}   кДж/моль")
for base, acid in pairs:
    B, BH = thermo[base], thermo[acid]
    dE = (B["E(ele)"] - BH["E(ele)"]) * hartree_2_kj 
    PA = (B["H"] - BH["H"]) * hartree_2_kj + H_proton
    GB = (B["G"] - BH["G"]) * hartree_2_kj + G_proton
    print(f"{base:<10}{dE:8.1f}{PA:8.1f}{GB:8.1f}{exp[base][0]:9.1f}{exp[base][1]:9.1f}")

**Экспериментальные данные** (298 K, кДж/моль):

| Молекула | Сродство к протону PA | Газофазная основность GB |
|---|---:|---:|
| NH₃ | 853.6 | 819.0 |
| CH₃NH₂ | 899.0 | 864.5 |
| (CH₃)₂NH | 929.5 | 896.5 |
| (CH₃)₃N | 948.9 | 918.1 |

Hunter E. P. L., Lias S. G. Evaluated Gas Phase Basicities and Proton Affinities of Molecules: An Update // J. Phys. Chem. Ref. Data. 1998. V. 27. P. 413–656. https://doi.org/10.1063/1.556018.


## 1.3 Константа кислотности

**Константа кислотности** описывает равновесие в растворе: 

$$ \mathrm{BH^+_{(aq)} \rightarrow B_{(aq)} + H^+_{(aq)}} \qquad\qquad pK_a = \frac{\Delta G^\circ_{aq}}{RT\ln 10} $$
**Термодинамический цикл.** Энергию Гиббса в растворе собирают из газофазной и энергии сольватации каждого участника:

$$ G_{aq}(\mathrm{X}) = G_{gas}(\mathrm{X}) + \Delta G_{solv}(\mathrm{X}) $$

Энергию сольватации оценивают в континуальной модели (Занятие 4, глава 3.5) как разность электронных энергий одноточечных расчётов в растворителе и в вакууме в одной и той же геометрии.

**Протон.** У протона в воде нет ни геометрии, ни орбиталей. Его энергия Гиббса складывается из трёх известных чисел:

$$ G_{aq}(\mathrm{H^+}) = \underbrace{G^\circ_{gas}(\mathrm{H^+})}_{-26.3} + \underbrace{\Delta G^*_{solv}(\mathrm{H^+})}_{-1112.5} + \underbrace{RT\ln 24.46}_{+7.9} = -1130.9\ \text{кДж/моль} $$

1. Посчитано в главе 1.2; 
2. Экспериментальная энергия гидратации протона (-1112.5 кДж/моль, https://doi.org/10.1021/jp063552y); 
3. Коэффициент перехода от стандартного состояния газа (1 атм, 24.46 л/моль) к стандартному состоянию раствора (1 моль/л). 
(Для B и BH⁺ эта поправка одинакова и в разности сокращается);

**Абсолютная схема:**

$$ pK_a = \frac{G_{aq}(\mathrm{B}) + G_{aq}(\mathrm{H^+}) - G_{aq}(\mathrm{BH^+})}{RT\ln 10} $$

In [ ]:
# Те же амины, одноточечный расчёт CPCM на оптимизированной геометрии в вакууме
main_path = os.path.join(REPO, "Занятия", "Занятие_6")

folders = {
    "NH3":     "nh3_neutral",
    "NH4+":    "nh4_cation",
    "MeNH2":   "menh2",
    "MeNH3+":  "menh3_cation",
    "Me2NH":   "me2nh",
    "Me2NH2+": "me2nh2_cation",
    "Me3N":    "me3n",
    "Me3NH+":  "me3nh_cation",
}

for label, folder in folders.items():
    charge = 1 if label.endswith("+") else 0
    name = folder + "_cpcm"
    work_path = os.path.join(main_path, name)
    os.makedirs(work_path, exist_ok=True)
    os.chdir(work_path)

    input_file = f"""\
    ! B3LYP def2-SVP CPCM(water)
    %pal nprocs 24 end

    * xyzfile {charge} 1 ../{folder}/{folder}.xyz
    """
    with open(f"{name}.inp", "w") as f:
        f.write(input_file)

    print("Расчёт:", name)
    !{ORCA} {name}.inp > {name}.out --oversubscribe

os.chdir(main_path)

In [ ]:
# Сборка энергий для pKa: газовая фаза (E, G из Freq) и вода (E из CPCM), хартри
main_path = os.path.join(REPO, "Занятия", "Занятие_6")

folders = {
    "NH3":     "nh3_neutral",
    "NH4+":    "nh4_cation",
    "MeNH2":   "menh2",
    "MeNH3+":  "menh3_cation",
    "Me2NH":   "me2nh",
    "Me2NH2+": "me2nh2_cation",
    "Me3N":    "me3n",
    "Me3NH+":  "me3nh_cation",
}

def read_out(path, labels):
    """Последние значения строк .out, начинающихся с заданных префиксов (первое число после префикса)"""
    values = {key: None for key in labels.values()}
    finished = False
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            for prefix, key in labels.items():
                if line.startswith(prefix):
                    values[key] = float(line[len(prefix):].replace("...", " ").split()[0])
            if "ORCA TERMINATED NORMALLY" in line:
                finished = True
    if not finished:
        print(f"{os.path.basename(path)} не завершён")
    return values

gas  = {"Electronic energy": "E_gas", "Final Gibbs free energy": "G_gas"}
cpcm = {"FINAL SINGLE POINT ENERGY": "E_cpcm"}

energies = {}
print("energies = {")
for label, folder in folders.items():
    out_gas  = os.path.join(main_path, folder, f"{folder}.out")
    out_cpcm = os.path.join(main_path, folder + "_cpcm", f"{folder}_cpcm.out")
    if not (os.path.isfile(out_gas) and os.path.isfile(out_cpcm)):
        print(f"    # {label}: нет {folder}/{folder}.out или {folder}_cpcm/{folder}_cpcm.out")
        continue
    energies[label] = v = {**read_out(out_gas, gas), **read_out(out_cpcm, cpcm)}
    print(f'    "{label}":'.ljust(15) + f'{{"E_gas": {v["E_gas"]}, "G_gas": {v["G_gas"]}, "E_cpcm": {v["E_cpcm"]}}},   # {folder}')
print("}")


In [ ]:
hartree_2_kj = 2625.4996
T = 298.15
R = 8.314462e-3                                      # кДж/(моль·K)
RT_ln10 = R * T * math.log(10)                       # 5.71 кДж/моль на единицу pKa

# Протон в воде: G_gas + ΔG_solv + переход 1 атм → 1 моль/л
G_proton_gas   = 2.5 * R * T - T * 0.10885           #  -26.3 кДж/моль (глава 1.2)
dG_solv_proton = -265.9 * 4.184                      # -1112.5 кДж/моль, Kelly, Cramer, Truhlar 2006
G_proton_aq    = G_proton_gas + dG_solv_proton + R * T * math.log(24.46)   # -1130.9 кДж/моль

energies = {
    "NH3":     {"E_gas": None, "G_gas": None, "E_cpcm": None},   # nh3_neutral
    "NH4+":    {"E_gas": None, "G_gas": None, "E_cpcm": None},   # nh4_cation
    "MeNH2":   {"E_gas": None, "G_gas": None, "E_cpcm": None},   # menh2
    "MeNH3+":  {"E_gas": None, "G_gas": None, "E_cpcm": None},   # menh3_cation
    "Me2NH":   {"E_gas": None, "G_gas": None, "E_cpcm": None},   # me2nh
    "Me2NH2+": {"E_gas": None, "G_gas": None, "E_cpcm": None},   # me2nh2_cation
    "Me3N":    {"E_gas": None, "G_gas": None, "E_cpcm": None},   # me3n
    "Me3NH+":  {"E_gas": None, "G_gas": None, "E_cpcm": None},   # me3nh_cation
}

pairs = [("NH3", "NH4+"), ("MeNH2", "MeNH3+"), ("Me2NH", "Me2NH2+"), ("Me3N", "Me3NH+")]

def G_aq(label):
    """G в воде, кДж/моль: G_gas из Freq + ΔG_solv как разность одноточечных энергий. Возвращает (G_aq, ΔG_solv)"""
    e = energies[label]
    dG_solv = (e["E_cpcm"] - e["E_gas"]) * hartree_2_kj
    return e["G_gas"] * hartree_2_kj + dG_solv, dG_solv

pKa_calc = {}
print(f"{'основание':<10}{'ΔGsolv B':>10}{'ΔGsolv BH+':>12}{'ΔG_aq':>8}{'pKa':>7}")
for base, acid in pairs:
    G_B,  dG_B  = G_aq(base)
    G_BH, dG_BH = G_aq(acid)
    dG_aq = G_B + G_proton_aq - G_BH                     # BH+ → B + H+
    pKa_calc[base] = dG_aq / RT_ln10
    print(f"{base:<10}{dG_B:10.1f}{dG_BH:12.1f}{dG_aq:8.1f}{pKa_calc[base]:7.2f}")
print("энергии в кДж/моль")


| Основание | $pK_a(\mathrm{BH^+})$ | $pK_b$ |
|---|---:|---:|
| NH₃ | 9.25 | 4.75 |
| CH₃NH₂ | 10.66 | 3.34 |
| (CH₃)₂NH | 10.73 | 3.27 |
| (CH₃)₃N | 9.80 | 4.20 |

Haynes W. M. (ed.) CRC Handbook of Chemistry and Physics, 97th ed. CRC Press, 2016. Section 5: Dissociation Constants of Organic Acids and Bases; Dissociation Constants of Inorganic Acids and Bases.

In [ ]:
COLORS = ["#000000", "#D64933"]

pKa_exp = {"NH3": 9.25, "MeNH2": 10.66, "Me2NH": 10.73, "Me3N": 9.80}
bases = list(pKa_calc)
x = np.array([pKa_calc[b] for b in bases])
y = np.array([pKa_exp[b]  for b in bases])
r2 = lambda y, y_pred: 1 - np.sum((y - y_pred)**2) / np.sum((y - y.mean())**2)

a, b = np.polyfit(x, y, 1)  # линейная подгонка pKa_exp = a·pKa_calc + b
x_corr = a * x + b

fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.7))
for ax, xs, xlabel in zip(axes, [x, x_corr], ["pKa, расчёт", "pKa, расчёт после подгонки"]):
    lo, hi = min(xs.min(), y.min()) - 0.5, max(xs.max(), y.max()) + 0.5
    ax.plot([lo, hi], [lo, hi], color=COLORS[0], lw=0.6, ls="--")
    ax.plot(xs, y, "o", color=COLORS[0])
    for name, xi, yi in zip(bases, xs, y):
        up = yi > xi
        ax.annotate(name, (xi, yi), xytext=(-4, 4) if up else (4, -8),
            textcoords="offset points", ha="right" if up else "left", fontsize=6)
    ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel=xlabel)
    ax.text(0.04, 0.96, f"R² = {r2(y, xs):.2f}", color=COLORS[0], transform=ax.transAxes, va="top", fontsize=8)
axes[0].set_ylabel("pKa, эксперимент")
xx = np.array(axes[0].get_xlim())
axes[0].plot(xx, a * xx + b, color=COLORS[1])
axes[0].text(0.04, 0.89, f"y = {a:.2f}x + {b:.2f},  R² = {r2(y, x_corr):.2f}", color=COLORS[1],
             transform=axes[0].transAxes, va="top", fontsize=8)
plt.show()

## 1.4 Окислительно-восстановительный потенциал

**Стандартный потенциал** полуреакции $\mathrm{Ox} + n\,e^- \rightarrow \mathrm{Red}$ - может быть рассчитан с использованием энергии Гиббса реакции восстановления в растворе, пересчитанную на один электрон и отнесённую к электроду сравнения:

$$ E^\circ = -\frac{\Delta G^\circ_{aq}}{nF} - E_{abs}(\mathrm{SHE}) $$

Принятое значение водородного электрода для использования в теоретических расчетах $E_{abs}(\mathrm{SHE}) = 4.28$ В (https://doi.org/10.1021/jp100402x)

**Хинон/гидрохинон.** $\mathrm{Q + 2H^+ + 2e^- \rightarrow H_2Q}$

Энергия Гиббса реакции:
$$ \Delta G_{\mathrm{red}} = G_{aq}(\mathrm{H_2Q}) - G_{aq}(\mathrm{Q}) - 2\,G_{aq}(\mathrm{H^+}), \qquad G(e^-) = 0 $$

Абсолютный потенциал реакции:
$$ E_{\mathrm{abs}} = -\frac{\Delta G_{\mathrm{red}}}{nF} $$

Потенциал реакции относительно электрода сравнения:
$$ E^\circ = E_{\mathrm{abs}} - E_{\mathrm{abs}}(\mathrm{SHE})


In [ ]:
# Хинон и гидрохинон
main_path = os.path.join(REPO, "Занятия", "Занятие_6")

species = {
    # папка           заряд  геометрия
    "benzoquinone": (0, """\
O    2.597174   0.567847  -0.000572
C    1.402180   0.306573  -0.000309
C    0.383364   1.375102  -0.007414
C   -0.922260   1.089640  -0.007127
C   -1.402180  -0.306573   0.000309
O   -2.597173  -0.567848   0.000572
C   -0.383364  -1.375102   0.007414
C    0.922260  -1.089640   0.007125
H    0.752051   2.393711  -0.012819
H   -1.682405   1.861441  -0.012284
H   -0.752052  -2.393711   0.012821
H    1.682405  -1.861440   0.012283
"""),
    "hydroquinone": (0, """\
O   -2.742737   0.075656   0.133789
C   -1.382833   0.020180   0.052554
C   -0.742385  -0.923190  -0.746459
C    0.652141  -0.941106  -0.797442
C    1.382833  -0.020179  -0.052555
O    2.742738  -0.075654  -0.133791
C    0.742385   0.923188   0.746461
C   -0.652141   0.941107   0.797441
H   -3.114177  -0.619831  -0.433422
H   -1.303831  -1.644869  -1.330472
H    1.169021  -1.669881  -1.415274
H    3.114177   0.619831   0.433423
H    1.303831   1.644866   1.330477
H   -1.169021   1.669883   1.415271
"""),
}

for name, (charge, xyz) in species.items():
    work_path = os.path.join(main_path, name)
    os.makedirs(work_path, exist_ok=True)
    os.chdir(work_path)

    input_file = f"""\
! B3LYP def2-SVP Opt Freq
%pal nprocs 24 end

* xyz {charge} 1
{xyz}*
"""
    with open(f"{name}.inp", "w") as f:
        f.write(input_file)

    print("Расчёт:", name)
    !{ORCA} {name}.inp > {name}.out --oversubscribe

os.chdir(main_path)

In [ ]:
# Хинон и гидрохинон: одноточечный расчёт CPCM(water) на газофазной геометрии
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
redox_folders = {"Q": "benzoquinone", "H2Q": "hydroquinone"}

for label, folder in redox_folders.items():
    name = folder + "_cpcm"
    work_path = os.path.join(main_path, name)
    os.makedirs(work_path, exist_ok=True)
    os.chdir(work_path)

    input_file = f"""\
! B3LYP def2-SVP CPCM(water)
%pal nprocs 24 end

* xyzfile 0 1 ../{folder}/{folder}.xyz
"""
    with open(f"{name}.inp", "w") as f:
        f.write(input_file)

    print("Расчёт:", name)
    !{ORCA} {name}.inp > {name}.out --oversubscribe

os.chdir(main_path)

In [ ]:
# Сборка энергий
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
redox_folders = {"Q": "benzoquinone", "H2Q": "hydroquinone"}

def energies_from_out(path, labels):
    values = {key: None for key in labels.values()}
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            for prefix, key in labels.items():
                if line.startswith(prefix):
                    values[key] = float(line[len(prefix):].replace("...", " ").split()[0])
    return values

gas  = {"Electronic energy": "E_gas", "Final Gibbs free energy": "G_gas"} 
cpcm = {"FINAL SINGLE POINT ENERGY": "E_cpcm"}

energies_redox = {}
print("energies_redox = {")
for label, folder in redox_folders.items():
    out_gas  = os.path.join(main_path, folder, f"{folder}.out")
    out_cpcm = os.path.join(main_path, folder + "_cpcm", f"{folder}_cpcm.out")
    if not (os.path.isfile(out_gas) and os.path.isfile(out_cpcm)):
        print(f"    # {label}: нет {folder}/{folder}.out или {folder}_cpcm/{folder}_cpcm.out")
        continue
    energies_redox[label] = v = {**energies_from_out(out_gas, gas), **energies_from_out(out_cpcm, cpcm)}
    print(f'    "{label}":'.ljust(11) + f'{{"E_gas": {v["E_gas"]}, "G_gas": {v["G_gas"]}, "E_cpcm": {v["E_cpcm"]}}},   # {folder}')
print("}")

In [ ]:
hartree_2_kj = 2625.4996
T = 298.15
R = 8.314462e-3                                      # кДж/(моль·K)
F = 96.485                                           # кДж/(моль·В)
nernst = R * T * math.log(10) / F                    # 0.0592 В на единицу pH

G_proton_aq = (2.5 * R * T - T * 0.10885) + (-265.9 * 4.184) + R * T * math.log(24.46)   # -1130.9 кДж/моль
E_SHE_abs   = 4.28                                   # В, абсолютный потенциал SHE
n = 2                                                # электронов: Q + 2H+ + 2e- → H2Q

energies_redox = {
    "Q":   {"E_gas": None, "G_gas": None, "E_cpcm": None},   # benzoquinone
    "H2Q": {"E_gas": None, "G_gas": None, "E_cpcm": None},   # hydroquinone
}

def G_aq(label):
    e = energies_redox[label]
    dG_solv = (e["E_cpcm"] - e["E_gas"]) * hartree_2_kj
    return e["G_gas"] * hartree_2_kj + dG_solv, dG_solv

G_Q,   dG_Q   = G_aq("Q")
G_H2Q, dG_H2Q = G_aq("H2Q")
dG_red = G_H2Q - G_Q - 2 * G_proton_aq
E_abs  = -dG_red / (n * F)
E0     = E_abs - E_SHE_abs

print(f"ΔG_solv(Q)   = {dG_Q:7.1f} кДж/моль")
print(f"ΔG_solv(H2Q) = {dG_H2Q:7.1f} кДж/моль")
print(f"ΔG_red       = {dG_red:7.1f} кДж/моль") 
print(f"E_abs        = {E_abs:7.3f} В")
print(f"E°           = {E0:7.3f} В vs SHE")

Экспериментальные значения (https://doi.org/10.1021/ja01673a014): 

| Величина | Эксперимент |
|---|---:|
| $E^\circ$ (Q/H₂Q) vs SHE | 0.699 В |
| $\Delta G_{\mathrm{red}}$ | −960.8 кДж/моль |

# ⚛ Глава 2. Электрические и магнитные свойства

## 2.1 Дипольный момент

Дипольный момент характеризует пространственное разделение положительного и отрицательного зарядов в молекуле. Это векторная величина, зависящая как от распределения электронной плотности, так и от геометрии молекулы.

В квантово-химических расчётах дипольный момент определяется суммой вкладов ядер и электронной плотности. 
$$ \boldsymbol{\mu} = \sum_A Z_A \mathbf{R}_A - \int \rho(\mathbf{r})\,\mathbf{r}\,d\mathbf{r} $$

Модуль дипольного момента обычно выражают в дебаях (D). Дипольный момент можно определить экспериментально, поэтому его удобно использовать для проверки качества квантово-химических методов. 

In [ ]:
# H2O (HF)
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "dip_h2o_hf")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! HF ma-def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 1
O   0.000000   0.000000   0.000000
H   0.000000   0.756950   0.585882
H   0.000000  -0.756950   0.585882
*
"""
with open("dip_h2o_hf.inp", "w") as f:
    f.write(input_file)

!{ORCA} dip_h2o_hf.inp > dip_h2o_hf.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("dip_h2o_hf/dip_h2o_hf.out")

In [ ]:
# H2O (B3LYP)
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "dip_h2o_b3lyp")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP ma-def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 1
O   0.000000   0.000000   0.000000
H   0.000000   0.756950   0.585882
H   0.000000  -0.756950   0.585882
*
"""
with open("dip_h2o_b3lyp.inp", "w") as f:
    f.write(input_file)

!{ORCA} dip_h2o_b3lyp.inp > dip_h2o_b3lyp.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("dip_h2o_b3lyp/dip_h2o_b3lyp.out")

In [ ]:
# CO (HF)
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "dip_co_hf")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! HF ma-def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 1
C   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.128300
*
"""
with open("dip_co_hf.inp", "w") as f:
    f.write(input_file)

!{ORCA} dip_co_hf.inp > dip_co_hf.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("dip_co_hf/dip_co_hf.out")

In [ ]:
# CO (B3LYP)
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "dip_co_b3lyp")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP ma-def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 1
C   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.128300
*
"""
with open("dip_co_b3lyp.inp", "w") as f:
    f.write(input_file)

!{ORCA} dip_co_b3lyp.inp > dip_co_b3lyp.out --oversubscribe
os.chdir(main_path)


In [ ]:
show("dip_co_b3lyp/dip_co_b3lyp.out")

| Молекула | $\mu$, Д | Источник |
|---|---:|---|
| H₂O | 1.855 | https://doi.org/10.1063/1.1680328 |
| CO | 0.110 (C⁻O⁺) | https://doi.org/10.1016/0022-2852(75)90287-8 |

## 2.2 Поляризуемость

Поляризуемость характеризует способность электронной плотности молекулы изменяться под действием внешнего электрического поля. В отличие от постоянного дипольного момента, она описывает отклик молекулы на внешнее воздействие и может быть ненулевой даже у неполярных молекул.

В слабом электрическом поле индуцированный дипольный момент в первом приближении пропорционален напряжённости поля:

$$ \boldsymbol{\mu}(\mathbf{F}) = \boldsymbol{\mu}_0 + \boldsymbol{\alpha}\mathbf{F} $$

где $\boldsymbol{\mu}_0$ - постоянный дипольный момент, а $\boldsymbol{\alpha}$ - тензор поляризуемости.

Поляризуемость молекулы напрямую связана с рядом экспериментально измеряемых характеристик, включая **показатель преломления** и **интенсивность рамановского рассеяния**. При этом для описания взаимодействия с электромагнитным излучением необходимо учитывать зависимость поляризуемости от частоты поля. Для электронных спектров поглощения поляризуемость представляет дополнительную характеристику отклика электронной системы, тогда как интенсивность отдельных переходов определяется соответствующими переходными дипольными моментами.

Для расчёта тензора поляризуемости в Orca 6 можно использовать ключ 
`%elprop Polar 1 end`

## 2.3 Электростатический потенциал

**Электростатический потенциал** (ESP) описывает взаимодействие молекулы с пробным электрическим зарядом в различных точках пространства. В отличие от дипольного момента, который характеризует распределение зарядов одной векторной величиной, электростатический потенциал позволяет анализировать пространственное распределение электростатических свойств молекулы.

$$ V(\mathbf r) = \sum_A \frac{Z_A}{|\mathbf r - \mathbf R_A|} - \int \frac{\rho(\mathbf r')}{|\mathbf r - \mathbf r'|}\,d\mathbf r' $$

Для наглядного представления распределение электростатического потенциала строят на поверхности молекулы (обычно на расстояниях Ван-дер-Ваальса). Такие карты помогают интерпретировать возможные электростатические взаимодействия между молекулами, оценивать распределение электрофильных и нуклеофильных участков и анализировать потенциальные места взаимодействия с лигандами. 

In [ ]:
# H2O: оптимизация B3LYP/ma-def2-TZVP; кубы плотности и ESP строит show(esp=True)
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "esp_h2o")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP ma-def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 1
O   0.000000   0.000000   0.000000
H   0.000000   0.756950   0.585882
H   0.000000  -0.756950   0.585882
*
"""
with open("esp_h2o.inp", "w") as f:
    f.write(input_file)

!{ORCA} esp_h2o.inp > esp_h2o.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("esp_h2o/esp_h2o.out", esp=True)

In [ ]:
# H2CO
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "esp_h2co")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP ma-def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 1
C   0.000000   0.000000   0.000000
O   1.205000   0.000000   0.000000
H  -0.587640   0.942870   0.000000
H  -0.587640  -0.942870   0.000000
*
"""
with open("esp_h2co.inp", "w") as f:
    f.write(input_file)

!{ORCA} esp_h2co.inp > esp_h2co.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("esp_h2co/esp_h2co.out", esp=True)

# ♨️Глава 3. Колебательные свойства

## 3.1 Инфракрасная спектроскопия

Молекулы обладают набором нормальных колебаний, каждому из которых соответствует определённая частота. Энергии переходов между колебательными уровнями, как правило, лежат в области инфракрасного спектра. 

В гармоническом приближении частоты нормальных колебаний определяются из собственных значений матрицы Гессе, то есть матрицы вторых производных энергии по координатам ядер, с учётом масс атомов (Занятие 5, глава 2.1)

Основное правило отбора для ИК-спектроскопии: колебание активно в ИК-спектре, если при нём изменяется дипольный момент молекулы. Интенсивность полосы определяется величиной этого изменения. В гармоническом приближении интенсивность связана с производной дипольного момента по нормальной координате $Q_k$:

$$  I_k \propto \left|\frac{\partial \boldsymbol{\mu}}{\partial Q_k}\right|^2 $$

Таким образом, не каждое нормальное колебание обязательно проявляется в ИК-спектре. Например, симметричное растяжение связей C=O в линейной молекуле CO₂ не сопровождается изменением дипольного момента и поэтому ИК-неактивно, тогда как асимметричное растяжение активно.

In [ ]:
# CO2
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "co2")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP Opt Freq
%pal nprocs 24 end

%freq ScalFreq 0.9671 end

* xyz 0 1
O   0.000000   0.000000  -1.162000
C   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.162000
*
"""
with open("co2.inp", "w") as f:
    f.write(input_file)

!{ORCA} co2.inp > co2.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("co2/co2.out")

| № | Симметрия | ν, см⁻¹ | Активность в ИК |
|---|---|---:|---|
| 1 | Πᵤ | 667 | Да |
| 2 | Σg⁺ | 1333 | Нет |
| 3 | Σᵤ⁺ | 2349 | Да |

https://webbook.nist.gov/cgi/cbook.cgi?ID=C124389&Mask=800


## 3.2 Спектры комбинационного рассеяния

Рамановская (комбинационная) спектроскопия основана на неупругом рассеянии света молекулами. При взаимодействии с падающим излучением молекула может перейти на другой колебательный уровень, вследствие чего энергия рассеянного фотона отличается от энергии падающего. Разность частот соответствует частоте молекулярного колебания.

В отличие от ИК-спектроскопии, активность колебания в КР определяется изменением поляризуемости молекулы. Для нормальной координаты (Q_k) интенсивность рамановской полосы связана с производной тензора поляризуемости:

$$ I_k^{\mathrm{Raman}} \propto \left|\frac{\partial \boldsymbol{\alpha}}{\partial Q_k}\right|^2$$

Поэтому колебание, неактивное в ИК-спектре, может проявляться в КР-спектре, и наоборот. 

In [ ]:
# Бензол
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "benzene_freq")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! PBE0 def2-SVP Opt Freq
%pal nprocs 24 end

%elprop Polar 1 end
%freq ScalFreq 0.9555  end

* xyz 0 1
C   1.390000   0.000000   0.000000
C   0.695000   1.203775   0.000000
C  -0.695000   1.203775   0.000000
C  -1.390000   0.000000   0.000000
C  -0.695000  -1.203775   0.000000
C   0.695000  -1.203775   0.000000
H   2.470000   0.000000   0.000000
H   1.235000   2.139082   0.000000
H  -1.235000   2.139082   0.000000
H  -2.470000   0.000000   0.000000
H  -1.235000  -2.139082   0.000000
H   1.235000  -2.139082   0.000000
*
"""
with open("benzene_freq.inp", "w") as f:
    f.write(input_file)

!{ORCA} benzene_freq.inp > benzene_freq.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("benzene_freq/benzene_freq.out")

| № | Симметрия | ν, см⁻¹ | Активность |
|---|---|---:|---|
| 1 | E₂ᵤ | 398 | нет |
| 2 | E₂g | 608.1 | КР |
| 3 | A₂ᵤ | 674.0 | ИК |
| 4 | B₂g | 707 | нет |
| 5 | E₁g | 847.1 | КР |
| 6 | E₂ᵤ | 967 | нет |
| 7 | B₂g | 990 | нет |
| 8 | A₁g | 993.1 | КР |
| 9 | B₁ᵤ | 1010 | нет |
| 10 | E₁ᵤ | 1038.3 | ИК |
| 11 | B₂ᵤ | 1149.7 | нет |
| 12 | E₂g | 1177.8 | КР |
| 13 | B₂ᵤ | 1309.4 | нет |
| 14 | A₂g | 1350 | нет |
| 15 | E₁ᵤ | 1484.0 | ИК |
| 16 | E₂g | 1600.9 | КР |
| 17 | E₂g | 3056.7 | КР |
| 18 | B₁ᵤ | 3057 | нет |
| 19 | E₁ᵤ | 3064.4 | ИК |
| 20 | A₁g | 3073.9 | КР |

Ссылки на статьи с исходными данными:
* https://doi.org/10.1021/j100176a008
* https://doi.org/10.1063/1.450717
* https://doi.org/10.6028/nbs.nsrds.39


# 💫 Глава 4. Электронные свойства 

До этого мы работали только с основным электронным состоянием, но для изучения спектральных свойств молекул необходимо работать с возбуждёнными электронными состояниями. Квантово-химические методы позволяют рассчитывать энергии таких состояний, вероятности электронных переходов и соответствующие спектры.

Из основного электронного состояния с закрытой оболочкой $(S0)$ переносом электрона с занятой орбитали $i$ на вакантную $a$ можно получить два типа возбужденных состояний: **триплет** (спины параллельны) и **синглет** (спины антипараллельны). В методе Хартри-Фока низший триплет можно посчитать обычным методом самосогласованного поля для молекулы с мультиплетностью 3, а открытый синглет уже нет: нужны методы, работающие с суперпозициями конфигураций.

## 4.1 Метод конфигурационного взаимодействия? - Да!

В основном состоянии молекулы электроны размещаются на орбиталях в соответствии с принципом Паули и энергетическим порядком. При электронном возбуждении один или несколько электронов переходят на более высокие по энергии орбитали, поэтому для описания возбужденных состояний необходимо учитывать возможные электронные конфигурации и взаимодействие между ними. 

Простейшим способом получить энергии возбуждённых синглетов является метод конфигурационного взаимодействия с однократными возбуждениями CIS (раздел 2.2.3 занятия 4). Хотя однократно возбуждённые детерминанты не смешиваются (По теореме Бриллюэна) с детерминантом Хартри-Фока и CIS не улучшает основное состояние, зато его собственные векторы - возбуждённые состояния, а собственные значения - энергии возбуждения.

Но поскольку метод CIS учитывает только однократные возбуждения и не описывает электронную корреляцию в полной мере, энергии возбуждения, полученные методом CIS, получаются завышенными.

In [ ]:
# Акролеин
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "acrolein_hf")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! HF def2-SVP Opt
%pal nprocs 24 end

* xyz 0 1
C   -1.215866  -0.152692   0.000000
C   -0.057615   0.515740   0.000000
C    1.230901  -0.204319   0.000000
O    2.296551   0.398612   0.000000
H   -1.264797  -1.237135   0.000000
H   -2.158372   0.387098   0.000000
H   -0.007791   1.598271   0.000000
H    1.176989  -1.305575   0.000000
*
"""
with open("acrolein_hf.inp", "w") as f:
    f.write(input_file)

!{ORCA} acrolein_hf.inp > acrolein_hf.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("acrolein_hf/acrolein_hf.out")

In [ ]:
# Акролеин: CIS, 10 синглетных возбуждённых состояний в геометрии HF
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "acrolein_cis")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! HF def2-SVP PrintMOs PrintBasis
%pal nprocs 24 end
%cis
  NRoots 10
end

* xyzfile 0 1 ../acrolein_hf/acrolein_hf.xyz
"""
with open("acrolein_cis.inp", "w") as f:
    f.write(input_file)

!{ORCA} acrolein_cis.inp > acrolein_cis.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("acrolein_cis/acrolein_cis.out")

**Акролеин, электронные переходы** (газовая фаза):

| Переход | $E$, эВ | $\lambda$, нм | Интенсивность |
|---|---:|---:|---|
| $n$→$\pi$* | 3.71 | 334 | слабый, $\varepsilon \sim 10^1$ |
| $\pi$→$\pi$* | 6.41 | 193 | сильный, $\varepsilon \sim 10^4$ |

https://doi.org/10.1039/TF9454100498

## 4.2 Нестационарная теория функционала плотности

Нестационарная теория функционала плотности (Time-Dependent Density Functional Theory, TD-DFT) позволяет рассчитывать электронные возбуждения, рассматривая отклик электронной плотности на зависящее от времени внешнее возмущение (например, электрическое поле электромагнитной волны).

В таком случае эволюцию квантовой системы уже необходимо описывать с использованием **Временного уравнения Шрёдингера**:

$$ i\hbar\,\frac{\partial}{\partial t}\Psi(\mathbf r,t) = \hat H\,\Psi(\mathbf r,t) $$

Для метода TD-DFT существуют две важные теоремы, аналоги теорем стационарной DFT (Занятие 4, глава 1.3)

**Теорема Рунге-Гросса** (Аналог теоремы Хэнберга-Конна для нестационарной системы): 

Свойства эволюционирующей системы во времени для многоэлектронной системы определяются только её зависящей от времени электронной плотности $\rho{(x,y,z,t)}$, а электронная плотность однозначно задаёт зависящий от времени внешний потенциал системы и её полную энергию с точностью до поправки, зависящей только от времени (фазы). 

**Теорема ван Леувена** (Аналог теоремы Кона-Шэма):

Для каждой зависящей от времени плотности взаимодействующих частиц $\rho{(x,y,z,t)}$ существует потенциал невзаимодействующих частиц  $v_{ks}{(x,y,z,t)}$, дающий точно такую же зависящую от времени электронную плотность.

**Теория отклика**:

При взаимодействии молекулы с электромагнитным излучением внешнее электрическое поле периодически возмущает электронную систему. В режиме линейного отклика предполагается, что возмущение достаточно мало, а изменение электронной плотности пропорционально приложенному полю. Отклик системы зависит от частоты поля и особенно усиливается вблизи частот, соответствующих разрешённым электронным переходам.

Если мы ограничимся только линейным откликом системы, то решать нестационарное уравнение Кона-Шэма не нужно, будет достаточно найти частоты, соответствующие полюсам функции линейного отклика. В стандартной формулировке TD-DFT сводится к уравнениям Касиды:

$$ \begin{pmatrix}\mathbf A(\omega) & \mathbf B(\omega)\\ \mathbf B^*(\omega) & \mathbf A^*(\omega)\end{pmatrix}\begin{pmatrix}\mathbf X\\ \mathbf Y\end{pmatrix} = \Omega\begin{pmatrix}\mathbf I & 0\\ 0 & -\mathbf I\end{pmatrix}\begin{pmatrix}\mathbf X\\ \mathbf Y\end{pmatrix} $$

где $\Omega$ - набор частот переходов, а $X, Y$ - набор векторов возбуждений и девозбуждений. Матрицы $\mathbf A$ и $\mathbf B$ определяются энергиями орбиталей и взаимодействием между молекулярными орбиталями.

Если матрицу B приравнять к 0, то получаем Y = 0 - приближение Тамма-Данкова. В этом случае уравнение сводится к обычной задаче на собственные значения:

$$\mathbf A\mathbf X=\Omega\mathbf X$$

Это упрощает расчёт и в некоторых случаях улучшает численную устойчивость, хотя может изменять рассчитанные энергии возбуждения. 

Интенсивность электронного перехода принято выражать безразмерной **силой осциллятора**: 

$$ f_{0n} = \tfrac{2}{3}\,\omega_{0n}\,|\mathbf d_{0n}|^2 $$

где $\omega_{0n}$ показывает энергию перехода в а.е., а вектор $d$ - показывает дипольный момент перехода. Сумма сил осцилляторов по всем переходам равна числу электронов (правило сумм Томаса-Райхе-Куна), поэтому $f$ сильного разрешённого перехода порядка единицы. 

## 4.3 Спектр поглощения

In [ ]:
# Акролеин: оптимизация геометрии основного состояния
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "acrolein")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP Opt Freq
%pal nprocs 24 end

* xyz 0 1
C   -1.215866  -0.152692   0.000000
C   -0.057615   0.515740   0.000000
C    1.230901  -0.204319   0.000000
O    2.296551   0.398612   0.000000
H   -1.264797  -1.237135   0.000000
H   -2.158372   0.387098   0.000000
H   -0.007791   1.598271   0.000000
H    1.176989  -1.305575   0.000000
*
"""
with open("acrolein.inp", "w") as f:
    f.write(input_file)

!{ORCA} acrolein.inp > acrolein.out --oversubscribe
os.chdir(main_path)

In [ ]:
# Акролеин: TD-DFT, 10 синглетных возбуждённых состояний в геометрии S0
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "acrolein_tddft")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP PrintMOs PrintBasis
%pal nprocs 24 end

%tddft
  NRoots 10
  TDA false
end

* xyzfile 0 1 ../acrolein/acrolein.xyz
"""
with open("acrolein_tddft.inp", "w") as f:
    f.write(input_file)

!{ORCA} acrolein_tddft.inp > acrolein_tddft.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("acrolein_tddft")

**Акролеин, электронные переходы** (газовая фаза):

| Переход | $E$, эВ | $\lambda$, нм | Интенсивность |
|---|---:|---:|---|
| $n$→$\pi$* | 3.71 | 334 | слабый, $\varepsilon \sim 10^1$ |
| $\pi$→$\pi$* | 6.41 | 193 | сильный, $\varepsilon \sim 10^4$ |

https://doi.org/10.1039/TF9454100498

## 4.4 Флуоресценция и оптимизация возбужденных состояний

Флуоресценция это процесс испускания кванта света при переходе молекулы из возбуждённого электронного состояния в состояние с меньшей энергией. После поглощения фотона молекулой её геометрия может измениться вследствие перераспределения электронной плотности в возбуждённом состоянии. Поэтому энергия флуоресцентного перехода может отличаться от энергии поглощённого фотона, а разность между положениями полос поглощения и испускания называют стоксовым сдвигом.

Таким образом, для расчёта спектра поглощения необходимо использовать геометрию основного состояния, а для расчёта спектра флуоресценции оптимизированную геометрию возбуждённого состояния.

In [ ]:
# H2CO: оптимизация геометрии основного состояния
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "h2co_s0")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! PBE0 def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 1
C   0.000000   0.000000   0.000000
O   1.205000   0.000000   0.000000
H  -0.587640   0.942870   0.000000
H  -0.587640  -0.942870   0.000000
*
"""
with open("h2co_s0.inp", "w") as f:
    f.write(input_file)

!{ORCA} h2co_s0.inp > h2co_s0.out --oversubscribe
os.chdir(main_path)

In [ ]:
# H2CO: TD-DFT в геометрии S0, вертикальное поглощение
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "h2co_tddft")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! PBE0 ma-def2-TZVP PrintMOs PrintBasis
%pal nprocs 24 end
%tddft
  NRoots 5
  TDA false
end

* xyzfile 0 1 ../h2co_s0/h2co_s0.xyz
"""
with open("h2co_tddft.inp", "w") as f:
    f.write(input_file)

!{ORCA} h2co_tddft.inp > h2co_tddft.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("h2co_tddft/h2co_tddft.out")

In [ ]:
# H2CO: оптимизация S1 (IRoot 1)
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "h2co_s1")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! PBE0 ma-def2-TZVP Opt PrintMOs PrintBasis
%pal nprocs 24 end
%tddft
  NRoots 3
  IRoot 1
  TDA false
end

* xyz 0 1
C   0.000000   0.000000   0.000000
O   1.250000   0.000000   0.350000
H  -0.587640   0.942870   0.000000
H  -0.587640  -0.942870   0.000000
*
"""
with open("h2co_s1.inp", "w") as f:
    f.write(input_file)

!{ORCA} h2co_s1.inp > h2co_s1.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("h2co_s1/h2co_s1.out")

**Формальдегид, экспериментальные значения**:

| Переход | $E$, эВ (нм) |
|---|---:|
| $S_0 \to S_1$ | 3.495 (354.8) |
| $S_0 \to S_2$ | 7.09 (175) |
| $S_1 \to S_0$ | 3.495 (354.8) |
| $S_2 \to S_0$ | не наблюдается |

* https://doi.org/10.1146/annurev.pc.34.100183.000335
* https://doi.org/10.1063/1.434467

## 4.5 Спектроскопия кругового дихроизма

Спектроскопия кругового дихроизма (КД) основывается на разности поглощения молекулой света с левой и правой круговой поляризацией, $\Delta\varepsilon = \varepsilon_L - \varepsilon_R$. У ахиральных молекул $\Delta\varepsilon = 0$ для всех переходов, в то время как для пары энантиомеров спектры КД имеют одинаковые по модулю, но противоположные по знаку полосы при одинаковых условиях измерения. Расчёт КД спектров позволяет сопоставлять экспериментальные спектры с квантово-химическими моделями и использовать спектроскопические данные для анализа абсолютной конфигурации хиральных молекул.

In [ ]:
# (R)-карвон
# Геометрия предварительно оптимизирована
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "carvone_r")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! PBE0 def2-SVP CPCM(cyclohexane) PrintMOs PrintBasis
%pal nprocs 24 end
%tddft
  NRoots 5
  TDA false
end

* xyz 0 1
C    3.636690   0.692973   0.171453
C    2.183083   0.393865   0.009132
C    1.292173   1.278032  -0.458345
C   -0.170665   1.025342  -0.596479
C   -0.647477  -0.140309   0.273400
C    0.291549  -1.318494   0.074366
C    1.730676  -0.969435   0.358736
O    2.502581  -1.792338   0.812153
C   -2.108006  -0.449414   0.054944
C   -3.064605   0.637062   0.445266
C   -2.543595  -1.605828  -0.438376
H    4.244783   0.001594  -0.417538
H    3.944439   0.563966   1.212190
H    3.861740   1.714375  -0.137954
H    1.646499   2.260794  -0.763496
H   -0.713492   1.942329  -0.352004
H   -0.402081   0.813917  -1.650773
H   -0.547837   0.189780   1.318384
H    0.021610  -2.172863   0.697271
H    0.252302  -1.653445  -0.971440
H   -2.899278   0.946652   1.482461
H   -2.934070   1.529730  -0.173864
H   -4.099035   0.307201   0.343912
H   -1.873326  -2.408814  -0.719971
H   -3.604658  -1.786675  -0.573429
*
"""
with open("carvone_r.inp", "w") as f:
    f.write(input_file)

!{ORCA} carvone_r.inp > carvone_r.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("carvone_r/carvone_r.out", ECD=True)

In [ ]:
# (S)-карвон
# Геометрия предварительно оптимизирована
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "carvone_s")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! PBE0 def2-SVP CPCM(cyclohexane) PrintMOs PrintBasis
%pal nprocs 24 end
%tddft
  NRoots 5
  TDA false
end

* xyz 0 1
C    3.636690   0.692973  -0.171453
C    2.183083   0.393865  -0.009132
C    1.292173   1.278032   0.458345
C   -0.170665   1.025342   0.596479
C   -0.647477  -0.140309  -0.273400
C    0.291549  -1.318494  -0.074366
C    1.730676  -0.969435  -0.358736
O    2.502581  -1.792338  -0.812153
C   -2.108006  -0.449414  -0.054944
C   -3.064605   0.637062  -0.445266
C   -2.543595  -1.605828   0.438376
H    4.244783   0.001594   0.417538
H    3.944439   0.563966  -1.212190
H    3.861740   1.714375   0.137954
H    1.646499   2.260794   0.763496
H   -0.713492   1.942329   0.352004
H   -0.402081   0.813917   1.650773
H   -0.547837   0.189780  -1.318384
H    0.021610  -2.172863  -0.697271
H    0.252302  -1.653445   0.971440
H   -2.899278   0.946652  -1.482461
H   -2.934070   1.529730   0.173864
H   -4.099035   0.307201  -0.343912
H   -1.873326  -2.408814   0.719971
H   -3.604658  -1.786675   0.573429
*
"""
with open("carvone_s.inp", "w") as f:
    f.write(input_file)

!{ORCA} carvone_s.inp > carvone_s.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("carvone_s/carvone_s.out", ECD=True)

# 🧲 Глава 5. Магнитные свойства

## 5.1 Химические сдвиги ЯМР

Ядерный магнитный резонанс (ЯМР) основан на взаимодействии магнитных моментов ядер с внешним магнитным полем. Электроны молекулы частично экранируют ядро от приложенного поля, поэтому резонансная частота зависит от химического окружения ядра.

Гамильтониан ЯМР с учётом магнитного экранирования можно записать в виде: 
$$\hat{H}_{\mathrm{NMR}} = -\gamma_N \hbar \hat{\mathbf{I}} \cdot (\mathbf{1}-\boldsymbol{\sigma}) \cdot \mathbf{B}_0 $$

где $\gamma_N$ - гиромагнитное отношение ядра, $\hat{\mathbf{I}}$ - оператор ядерного спина, $\boldsymbol{\sigma}$ - тензор магнитного экранирования, а $\mathbf{B}_0$ - внешнее магнитное поле. Тензор $\boldsymbol{\sigma}$ учитывает зависимость экранирования от направления поля относительно молекулы. Его изотропная часть определяет основной вклад в химический сдвиг, наблюдаемый в растворах при быстром изотропном вращении молекул.

В квантово-химических расчётах химический сдвиг определяется через тензор магнитного экранирования $\sigma$. Обычно изотропное экранирование вычисляют как:

$$ \sigma_{\mathrm{iso}} = \frac{1}{3} \left(\sigma_{xx} + \sigma_{yy} + \sigma_{zz}\right) $$

а химический сдвиг относительно эталонного соединения приближённо выражают как

$$ \delta=\sigma_{\mathrm{ref}}-\sigma_{\mathrm{sample}} $$

Экранирование эталона считают тем же методом и в том же базисе. При вычитании двух значений часть систематической ошибки компенсируется, поэтому относительные химические сдвиги получаются точнее.

**Почему для ЯМР нужен специальный базис?** Для расчёта парамагнитной части экранирования используют оператор орбитального углового момента $\hat{\mathbf L}_N$, а также множитель $1/r_N^3$, который чувствителен к поведению p-функций вблизи ядра, там, где обычные базисы не имеют достаточно гауссовых функций. Специальные базисы pcSseg-n разработаны для расчётов магнитного экранирования и содержат дополнительные p-функции с большими экспонентами, а p-оболочка описывается набором независимых гауссовых функций. Это позволяет точнее описывать поведение электронных волновых функций вблизи ядра. При этом s-функции исходного базиса не изменяются. 

Ключ `NMR` автоматически включает GIAO (gauge-including atomic orbitals): без них результат в конечном базисе зависит от выбора начала отсчёта векторного потенциала, то есть от положения молекулы в системе координат.

In [ ]:
# Метанол: оптимизация геометрии, B3LYP/def2-TZVP
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "meoh")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 1
C   -0.047000   0.663000   0.000000
O   -0.047000  -0.758000   0.000000
H    0.864000  -1.049000   0.000000
H   -1.087000   0.976000   0.000000
H    0.449000   1.065000   0.890000
H    0.449000   1.065000  -0.890000
*
"""
with open("meoh.inp", "w") as f:
    f.write(input_file)

!{ORCA} meoh.inp > meoh.out --oversubscribe
os.chdir(main_path)

In [ ]:
# Метанол: экранирования ЯМР, B3LYP/pcSseg-2
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "meoh_nmr")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP pcSseg-2 NMR TightSCF
%pal nprocs 24 end

* xyzfile 0 1 ../meoh/meoh.xyz
"""
with open("meoh_nmr.inp", "w") as f:
    f.write(input_file)

!{ORCA} meoh_nmr.inp > meoh_nmr.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("meoh_nmr/meoh_nmr.out")

In [ ]:
# Тетраметилсилан (ТМС): оптимизация геометрии, B3LYP/def2-TZVP
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "tms")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 1
Si   0.000222   0.000223   0.000222
C    1.088662   1.088663   1.088663
H    0.486174   1.733627   1.733630
H    1.733628   0.486173   1.733626
H    1.733626   1.733629   0.486173
C    1.088514  -1.088310  -1.088311
H    0.487135  -1.733846  -1.733847
H    1.734081  -1.733898  -0.487010
H    1.734086  -0.487013  -1.733898
C   -1.088310   1.088514  -1.088311
H   -1.733846   0.487136  -1.733848
H   -1.733899   1.734083  -0.487012
H   -0.487010   1.734085  -1.733896
C   -1.088309  -1.088310   1.088514
H   -1.733846  -1.733848   0.487136
H   -1.733897  -0.487012   1.734084
H   -0.487010  -1.733896   1.734085
*
"""
with open("tms.inp", "w") as f:
    f.write(input_file)

!{ORCA} tms.inp > tms.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("tms/tms.out")

In [ ]:
# ТМС: экранирования ЯМР, B3LYP/pcSseg-2 (эталон для δ)
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "tms_nmr")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP pcSseg-2 NMR TightSCF
%pal nprocs 24 end

* xyzfile 0 1 ../tms/tms.xyz
"""
with open("tms_nmr.inp", "w") as f:
    f.write(input_file)

!{ORCA} tms_nmr.inp > tms_nmr.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("meoh_nmr/meoh_nmr.out", nmr_ref="tms_nmr/tms_nmr.out")

**Метанол**. Экспериментальные данные: 
| Ядро | $\delta$, ppm (CDCl₃) |
|---|---:|
| ¹H, CH₃ | 3.49 |
| ¹H, OH | 1.09 |
| ¹³C | 50.41 |

https://doi.org/10.1021/jo971176v

## 5.2 Тензоры ЭПР

Электронный парамагнитный резонанс (ЭПР) исследует системы с неспаренными электронами. Их магнитное поведение определяется, в частности, электронным $\mathbf g$-тензором, который связывает спиновый момент с внешним магнитным полем, и тензором сверхтонкого взаимодействия - $\mathbf A$, описывающим взаимодействие электронного спина с ядерными спинами.

$$\hat{H}_{\mathrm{EPR}} = \mu_B \mathbf{S} \cdot \mathbf{g} \cdot \mathbf{B}_0 + \mathbf{S} \cdot \mathbf{A} \cdot \mathbf{I} $$

Для расчёта сверхтонких констант ЭПР нужна другая гибкость базисного набора. Контактное взаимодействие Ферми определяется спиновой плотностью непосредственно на ядре и описывается с помощью дельта-функции $\delta(\mathbf r_N)$. Поэтому для таких расчётов особенно важны компактные s-функции, позволяющие точнее описать спиновую плотность вблизи ядра. Для расчётов ЭПР разработаны специальные базисы, например EPR-II и EPR-III. 

In [ ]:
# CH3: оптимизация геометрии радикала
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "ch3")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 2
C   0.000000   0.000000   0.000000
H   1.079000   0.000000   0.000000
H  -0.539500   0.934441   0.000000
H  -0.539500  -0.934441   0.000000
*
"""
with open("ch3.inp", "w") as f:
    f.write(input_file)

!{ORCA} ch3.inp > ch3.out --oversubscribe
os.chdir(main_path)

In [ ]:
# CH3: g-тензор и константы сверхтонкого взаимодействия
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "ch3_epr")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! UKS B3LYP EPR-III TightSCF
%pal nprocs 24 end

* xyzfile 0 2 ../ch3/ch3.xyz

%eprnmr
  gtensor 1
  Nuclei = all C { aiso, adip }
  Nuclei = all H { aiso, adip }
end

"""
with open("ch3_epr.inp", "w") as f:
    f.write(input_file)

!{ORCA} ch3_epr.inp > ch3_epr.out --oversubscribe
os.chdir(main_path)

In [ ]:
show("ch3_epr/ch3_epr.out")

## 5.3 Расщепление в нулевом поле

В системах со спином $S>1/2$ взаимодействия между неспаренными электронами, а также спин-орбитальное взаимодействие могут приводить к зависимости энергии от ориентации спина относительно молекулярных осей. В отсутствие внешнего магнитного поля это вызывает расщепление спиновых уровней, называемое расщеплением в нулевом поле (zero-field splitting, ZFS):

$$ \hat{H}_{\mathrm{ZFS}} = D \left[S_z^2-\frac{S(S+1)}{3}\right] + E(S_x^2-S_y^2) $$

Его описывают параметрами $D$ и $E$, характеризующими аксиальную и ромбическую составляющие анизотропии.

In [ ]:
# CH2: оптимизация геометрии триплета
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "ch2_triplet")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP Opt
%pal nprocs 24 end

* xyz 0 3
C   0.000000   0.000000   0.000000
H   0.991000   0.000000  -0.422000
H  -0.991000   0.000000  -0.422000
*
"""
with open("ch2_triplet.inp", "w") as f:
    f.write(input_file)

!{ORCA} ch2_triplet.inp > ch2_triplet.out --oversubscribe
os.chdir(main_path)

In [ ]:
# CH2: тензор расщепления в нулевом поле, спин-спиновый и спин-орбитальный вклады
main_path = os.path.join(REPO, "Занятия", "Занятие_6")
work_path = os.path.join(main_path, "ch2_zfs")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP TightSCF
%pal nprocs 24 end

* xyzfile 0 3 ../ch2_triplet/ch2_triplet.xyz

%eprnmr
  DTensor ssandso
  DSS uno
  DSOC qro
end
"""
with open("ch2_zfs.inp", "w") as f:
    f.write(input_file)

!{ORCA} ch2_zfs.inp > ch2_zfs.out --oversubscribe
os.chdir(main_path)


In [ ]:
show("ch2_zfs/ch2_zfs.out")

**Триплетный метилен**
| Параметр | Эксперимент, см⁻¹ |
|---|---:|
| $D$ | 0.69 |
| $E$ | 0.003 |

https://doi.org/10.1063/1.1674129
